这份程序 scoreCalc.ipynb 的主要功能是对时间序列预测模型的预测结果进行异常评分、判定与可视化分析。它将模型预测值与真实值之间的残差转化为“异常分值”，并通过阈值自动判定异常，最后将分析结果保存。
以下是该程序的详细功能说明：
1. 实验环境与路径管理
自动定位实验结果：程序具备智能路径搜索功能 (find_latest_run_folder)。它会自动在指定的根目录下递归搜索 Parameters.json 文件，并根据最后修改时间，自动定位到最新一次运行的实验文件夹，无需人工指定具体路径。
参数归档读取：自动加载实验的超参数配置 (Parameters.json)，方便用户在后续分析中溯源模型设置。
2. 数据准备与校验
数据加载：读取实验保存的真实值 (y_true.npy) 和模型预测值 (y_pred.npy)。
维度一致性校验：通过 shape 和 ndim 检测，确保预测结果维度（[样本数, 节点数, 预测步长]）符合异常检测算法的输入要求，防止后续计算产生维度错位。
3. 核心异常评分算法 (compute_anomaly_score)
这是程序的核心，将“预测误差”转化为“异常分值”：
残差计算：首先计算预测值与真实值之间的绝对误差 (|y_true - y_pred|)。
维度聚合：
Horizon 聚合：通过 max 或 mean 处理预测时段内的误差，得到每个变量在当前窗口的误差值。
标准化 (Z-score)：计算所有样本中该变量误差的均值 (mu) 和标准差 (sigma)，对误差进行标准化，消除不同传感器量纲不同的影响。
变量聚合：将标准化后的变量误差再次聚合，得到最终的窗口级异常得分 (score)。
4. 阈值判定与标签生成
阈值策略：支持两种模式确定报警线：
百分位法 (percentile)：如设定 99% 分位数，即认为得分最高的 1% 为异常。
均值标准差法 (mean_std)：基于 3-sigma 原则，超过“均值 + 3倍标准差”的区域被视为异常。
异常判定：根据选定的阈值生成二值化的 pred_label（1 表示异常，0 表示正常），并统计异常样本的计数和占比。
5. 可视化与结果导出
异常曲线图：绘制异常分值随时间轴变化的折线图，并在图中叠加判定阈值红线，直观展示模型在哪些时间窗口判定为异常。
数据持久化：
将计算出的异常分值 (anomaly_score.npy) 和 判定标签 (anomaly_label.npy) 导出为 numpy 数组。
将包含详细索引、得分和标签的完整结果汇总保存为 CSV 文件 (anomaly_score.csv)，便于后续使用 Excel 或其他软件进行进一步评估（如计算召回率、误报率等）。

In [1]:
# ==========================================
# 1. 基础导入与配置环境
# ==========================================
from pathlib import Path              # 导入路径操作模块，用于跨平台文件路径管理
import json                           # 导入 JSON 模块，用于读取配置文件
import os                             # 导入 OS 模块，用于文件与目录系统操作
import numpy as np                    # 导入 NumPy，用于高效的数值矩阵运算
import pandas as pd                   # 导入 Pandas，用于处理表格化实验结果
import matplotlib.pyplot as plt       # 导入绘图库，用于结果可视化

# 更新 matplotlib 的全局默认参数，确保图表符合科研论文规范
plt.rcParams.update({
    'font.size': 14,                  # 设置全局字体大小为 14pt
    'font.family': ['Times New Roman', 'SimSun'] # 优先使用 Times New Roman 英文，SimSun 中文
})

In [6]:
# ==========================================
# 2. 定位并检查实验结果目录
# ==========================================
# 使用 Path 对象指定实验数据存放的根目录
save_folder = Path(r'D:/大学/博士论文/GNN部分/07_RA-STGNN/Test')  # 注意使用原始字符串避免路径分隔符问题

def find_latest_run_folder(root: Path) -> Path:
    """定义一个辅助函数，用于自动寻找目录下最近一次修改的实验文件夹"""
    candidates = [] # 初始化候选路径列表
    for p in root.rglob('Parameters.json'): # 递归查找所有子目录下的 Parameters.json 文件
        candidates.append(p.parent) # 获取配置文件的父级文件夹路径
    if not candidates: # 若未找到任何配置文件，抛出错误
        raise FileNotFoundError(f'No Parameters.json found under: {root}')
    # 按文件系统的最后修改时间 (st_mtime) 降序排序，取最新的一个
    candidates = sorted(candidates, key=lambda x: x.stat().st_mtime, reverse=True)
    return candidates[0] # 返回最新实验文件夹的 Path 对象

# 检查当前指定目录，如果不是文件夹或者里面没有配置记录，则自动切换到最新的实验运行目录
if save_folder.is_dir() and not (save_folder / 'Parameters.json').exists():
    save_folder = find_latest_run_folder(save_folder)

# 打印最终确认的实验路径和关键文件状态，用于调试
print('save_folder =', save_folder)
print('Parameters.json exists:', (save_folder / 'Parameters.json').exists())
print('y_true.npy exists:', (save_folder / 'y_true.npy').exists())
print('y_pred.npy exists:', (save_folder / 'y_pred.npy').exists())
print('model.pth exists:', (save_folder / 'model.pth').exists())

FileNotFoundError: No Parameters.json found under: D:\大学\博士论文\GNN部分\07_RA-STGNN\Test

In [ ]:
# ==========================================
# 3. 加载实验参数与原始数据
# ==========================================
params_path = save_folder / 'Parameters.json' # 构建配置文件路径
with params_path.open('r', encoding='utf-8') as f: # 以只读模式打开文件
    params = json.load(f) # 将 JSON 数据解析为 Python 字典

params_df = pd.DataFrame(sorted(params.items()), columns=['key', 'value']) # 将参数转化为 DataFrame 以便直观查看
params_df.head(20) # 显示前 20 个配置参数

# 加载预测阶段保存的真实标签与模型预测结果数组
y_true = np.load(save_folder / 'y_true.npy') # 加载真实值 (Shape: [sample_num, node_num, horizon_size])
y_pred = np.load(save_folder / 'y_pred.npy') # 加载模型输出预测值

print('y_true shape:', y_true.shape) # 打印真实值维度，用于验证
print('y_pred shape:', y_pred.shape) # 打印预测值维度

# 检查维度一致性，防止计算时产生广播错误
if y_true.shape != y_pred.shape:
    raise ValueError(f'shape mismatch: y_true {y_true.shape}, y_pred {y_pred.shape}')
# 检查是否为三维时间序列数据
if y_true.ndim != 3:
    raise ValueError('Expected saved outputs with shape [sample_num, node_num, horizon_size].')

由于当前只保存了最终预测结果，这里将异常分值定义为预测误差的聚合形式：

$$E_{t,i,h} = |y^{true}_{t,i,h} - y^{pred}_{t,i,h}|$$

先在 horizon 维度聚合，再对每个变量做标准化，最后在变量维度上取最大值，得到窗口级异常分值 $S_t$。

In [ ]:
# ==========================================
# 4. 异常检测评分算法逻辑
# ==========================================
def compute_anomaly_score(y_true: np.ndarray, y_pred: np.ndarray, horizon_reduce: str = 'max', var_reduce: str = 'max'):
    """该函数实现了一种聚合误差的方法，将多步预测的残差转化为单一的时序异常分值"""
    if y_true.shape != y_pred.shape: # 再次确保形状一致
        raise ValueError('y_true and y_pred must have the same shape.')

    error = np.abs(y_true - y_pred) # 计算每一时刻、每一变量、每一预测步长的绝对误差残差

    # horizon 维聚合：在预测时段内取最大值(最严重的残差)或平均残差
    if horizon_reduce == 'max':
        var_error = error.max(axis=2) # 沿 horizon 轴计算最大值
    elif horizon_reduce == 'mean':
        var_error = error.mean(axis=2) # 沿 horizon 轴计算平均值
    else:
        raise ValueError("horizon_reduce must be 'max' or 'mean'")

    # 标准化：计算训练集统计特征，将变量误差归一化到统一量纲(Z-Score)
    mu = var_error.mean(axis=0, keepdims=True) # 计算全局均值
    sigma = var_error.std(axis=0, keepdims=True) # 计算全局标准差
    sigma = np.where(sigma == 0, 1e-8, sigma) # 防止除零导致 NaN，设置极小值
    norm_var_error = (var_error - mu) / sigma # 执行标准化

    # 变量维聚合：在所有传感器变量中取最大/平均得分，作为该时间窗口的最终异常分数
    if var_reduce == 'max':
        score = norm_var_error.max(axis=1) # 沿变量轴计算最大值
    elif var_reduce == 'mean':
        score = norm_var_error.mean(axis=1) # 沿变量轴计算平均值
    else:
        raise ValueError("var_reduce must be 'max' or 'mean'")

    return { # 返回包含过程数据的字典
        'error_tensor': error, 'var_error': var_error, 'norm_var_error': norm_var_error,
        'score': score, 'mu': mu.squeeze(0), 'sigma': sigma.squeeze(0),
    }

# 调用评分计算函数
result = compute_anomaly_score(y_true, y_pred, horizon_reduce='max', var_reduce='max')
score = result['score'] # 提取最终的异常分值数组

print('score shape:', score.shape) # 打印分值维度
print('score min/max/mean:', float(score.min()), float(score.max()), float(score.mean())) # 打印得分统计信息

In [ ]:
# ==========================================
# 5. 异常判定与阈值选择
# ==========================================
def choose_threshold(scores: np.ndarray, method: str = 'percentile', percentile: float = 99.0):
    """定义阈值自动选择逻辑"""
    if method == 'percentile':
        return float(np.percentile(scores, percentile)) # 基于百分位（如 99% 处）确定阈值
    elif method == 'mean_std':
        return float(scores.mean() + 3 * scores.std()) # 基于 3-sigma 原则（概率统计异常）确定阈值
    else:
        raise ValueError("method must be 'percentile' or 'mean_std'")

threshold = choose_threshold(score, method='percentile', percentile=99.0) # 计算阈值
pred_label = (score > threshold).astype(int) # 将高于阈值的点标记为 1 (异常)，否则为 0 (正常)

print('threshold =', threshold) # 打印当前选定的阈值
print('anomaly count =', int(pred_label.sum())) # 打印异常点的总数
print('anomaly ratio =', float(pred_label.mean())) # 打印异常占比

In [ ]:
# ==========================================
# 6. 结果组织与可视化
# ==========================================
score_df = pd.DataFrame({ # 将处理后的结果整理为 DataFrame 方便后续查阅
    'sample_idx': np.arange(len(score)), # 时间轴索引
    'anomaly_score': score, # 对应的异常分值
    'pred_label': pred_label, # 最终的二值化预测标签
})

score_df.head(20) # 显示前 20 条数据

# 绘制异常得分的时间演化图
fig, ax = plt.subplots(1, 1, figsize=(18, 5), dpi=120) # 初始化 18x5 英寸的画布
ax.plot(score_df['sample_idx'], score_df['anomaly_score'], color='#1f77b4', linewidth=1.5, label='Anomaly Score') # 绘制折线图
ax.axhline(threshold, color='#d62728', linestyle='--', linewidth=2, label=f'Threshold = {threshold:.4f}') # 绘制红色虚线作为判定阈值
ax.set_xlabel('Window Index') # 设置 X 轴名称
ax.set_ylabel('Anomaly Score') # 设置 Y 轴名称
ax.set_title('Window-level anomaly score') # 设置图表标题
ax.legend() # 显示图例
ax.grid(alpha=0.25) # 添加半透明网格线
plt.tight_layout() # 自动调整布局以填充画布
plt.show() # 显示图片

In [ ]:
# ==========================================
# 7. 实验结果持久化
# ==========================================
score_save_path = save_folder / 'anomaly_score.npy' # 定义分值数组的保存路径
label_save_path = save_folder / 'anomaly_label.npy' # 定义标签数组的保存路径
csv_save_path = save_folder / 'anomaly_score.csv' # 定义 CSV 表格的保存路径

# 保存数据供后续绘图或评估使用
np.save(score_save_path, score) 
np.save(label_save_path, pred_label)
score_df.to_csv(csv_save_path, index=False, encoding='utf-8-sig') # 保存表格，encoding 指定 utf-8-sig 以便 Excel 打开

print('saved to:') # 打印保存路径确认
print(score_save_path)
print(label_save_path)
print(csv_save_path)